In [ ]:
#Resampling Strategies
from src.tracking import init, log_run
init()
from imblearn.over_sampling import SMOTE # using imblearn library to get synthetic minoprity oversampling technique uses nearest neighbour os minority class and crwates point betweem them
from imblearn.under_sampling import RandomUnderSampler # Undersamples the data by dropping majority class untilmwe get certain strategy ratio
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier

def xgb(**kw): # rather than defininf input we let user input everything
    return XGBClassifier(n_estimators = 400, max_depth=4, learning_rate = 0.1,
                         eval_metric = "aucpr",tree_method="hist",
                         n_jobs = 2, random_state=42, **kw)

log_run("xgb_smote",
        ImbPipeline([("smote",SMOTE(random_state = 42, k_neighbors = 5)),
                     ("clf",xgb())]),
        X_tr, y_tr, X_te, y_te, {"resample":"SMOTE"})


log_run("xgb_undersample",
        ImbPipeline([("smote",RandomUnderSampler(sampling_strategy = 0.1, random_state = 42)),
                     ("clf",xgb())]),
        X_tr, y_tr, X_te, y_te, {"resample":"RandomUnderSample_0.1"})

array([3.6720623e-05, 1.1118317e-04, 1.3734441e-03, ..., 3.1560729e-05,
       8.3164123e-06, 6.4614264e-04], shape=(56962,), dtype=float32)

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier

def xgb(**kw):
    return XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.1,
                         eval_metric="aucpr", tree_method="hist",
                         n_jobs=2, random_state=42, **kw)

spw = (y_tr == 0).sum() / (y_tr == 1).sum()

candidates = {
    "none":        xgb(),
    "pos_weight":  xgb(scale_pos_weight=spw),
    "undersample": ImbPipeline([("under", RandomUnderSampler(sampling_strategy=0.1, random_state=42)),
                                ("clf", xgb())]),
    "smote":       ImbPipeline([("smote", SMOTE(random_state=42, k_neighbors=5)),
                                ("clf", xgb())]),
}

cv = StratifiedKFold(5, shuffle=True, random_state=0)
results = {}

for label, model in candidates.items():
    s = cross_val_score(model, X_tr, y_tr, scoring="average_precision",
                        cv=cv, n_jobs=-1)
    results[label] = s
    print(f"{label:12s} {s.mean():.4f} ± {s.std():.4f}   folds: {np.round(s, 4)}")

best = max(results, key=lambda k: results[k].mean())
print(f"\nhighest mean: {best}")
for label, s in results.items():
    if label == best:
        continue
    gap = results[best].mean() - s.mean()
    noise = max(results[best].std(), s.std())
    print(f"  vs {label:12s} gap {gap:+.4f}  noise {noise:.4f}  "
          f"{'REAL' if gap > noise else 'tied — within noise'}")

none         0.8516 ± 0.0342   folds: [0.8785 0.8452 0.8812 0.8651 0.7882]
pos_weight   0.8476 ± 0.0289   folds: [0.8803 0.8517 0.8691 0.8396 0.7971]
undersample  0.7885 ± 0.0472   folds: [0.7293 0.8305 0.8329 0.8171 0.7328]
smote        0.8467 ± 0.0392   folds: [0.8986 0.8537 0.871  0.8258 0.7842]

highest mean: none
  vs pos_weight   gap +0.0041  noise 0.0342  tied — within noise
  vs undersample  gap +0.0631  noise 0.0472  REAL
  vs smote        gap +0.0049  noise 0.0392  tied — within noise


# Day 29 — Resampling strategies

## The trap I fell into

On a single train/test split, SMOTE had the best PR-AUC and ROC-AUC. Looked like a real finding. It wasn't.

A single split scores one particular set of frauds. A different split reshuffles the numbers. When two models differ by a few thousandths, one split can't tell you which is better — it's one exam separating students who scored 88 and 87.

*Rule:* if the gap between two models is smaller than the fold-to-fold spread, they're tied. Ranking them is reading noise.

---

## Results (5-fold CV, PR-AUC, on X_tr only)

| strategy | PR-AUC | folds |
|---|---|---|
| none | 0.8516 ± 0.0342 | 0.8785, 0.8452, 0.8812, 0.8651, 0.7882 |
| pos_weight | 0.8476 ± 0.0289 | 0.8803, 0.8517, 0.8691, 0.8396, 0.7971 |
| smote | 0.8467 ± 0.0392 | 0.8986, 0.8537, 0.8710, 0.8258, 0.7842 |
| undersample | 0.7885 ± 0.0472 | 0.7293, 0.8305, 0.8329, 0.8171, 0.7328 |

*none / pos_weight / SMOTE are tied. Undersampling is genuinely worse by ~0.063.*

Two things about how to read this:

*The ± is mostly fold difficulty, not method instability.* Every strategy craters on fold 5 (~0.73–0.80) and peaks on folds 1/3. They move together — so the ±0.03 measures which frauds landed in which fold. It overstates uncertainty about the differences between methods.

*So the real test is paired, fold by fold.* SMOTE beats none on 2 folds, loses on 3. pos_weight wins 3, loses 2. Signs flip → noise. Undersampling loses on *5 of 5* → real, even though the gap is only slightly above the ±.

---

## Why (teach-back)

*Why did the imbalance corrections do nothing?*
PR-AUC depends only on how transactions are ranked, not on absolute predicted scores. scale_pos_weight and SMOTE both mostly rescale probabilities and shift where a threshold would sit — and rescaling doesn't reorder. So on a rank-based metric they have little to bite on. Their real payoff is calibration and threshold choice → Day 36, not here. Boosted trees with enough depth also already find the minority regions unaided.

*Why doesn't SMOTE "learn what fraud looks like"?*
It interpolates between a minority point and its k nearest neighbours. Fraud here isn't one tight cluster but several distinct modes — so interpolating between two different modes lands in regions dense with legitimate transactions. Some synthetic points are plausible, some are mislabeled noise.

PCA doesn't change this: it's a linear transform, so a midpoint in component space is the same midpoint in original feature space. Two things that do follow from the PCA structure — components are variance-ordered, so V1–V3 dominate the Euclidean distances SMOTE's k-NN uses; and if Amount/Time sit unscaled in the matrix they swamp everything, making "nearest neighbour" effectively mean "similar transaction size."

*Why does undersampling hurt?*
sampling_strategy=0.1 cuts ~227k legitimate transactions down to ~4k. The model loses almost all information about what normal looks like and the boundary gets coarse. It also has the widest spread (±0.0472) — worse mean and less stable, because which rows get discarded matters.

---

## Method note

Resampling must live *inside* the imblearn pipeline so it's refit within each fold and never touches the validation fold. Resampling before cross_val_score leaks and inflates scores badly.

Compare on X_tr only — this is strategy selection, not final reporting. Test set stays untouched.

---

## Decision

Carry *pos_weight* forward. Same performance as none and SMOTE, no resampling step, no extra library in the serving path, and Optuna (Day 30) can tune it as a continuous knob.

For reports/model_comparison.md:

> On 5-fold CV, none / class-weighting / SMOTE were within noise of each other (0.847–0.852); random undersampling at 0.1 lost 0.063 consistently across all five folds. PR-AUC is rank-based, so re-weighting the loss doesn't move it much; discarding 98% of the majority class does.